In [5]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

rng = np.random.default_rng(123)
n = 18_000

segment = rng.choice(
    ['new', 'regular', 'vip'],
    size=n,
    p=[0.55, 0.30, 0.15]
)

variant = rng.integers(0, 2, size=n)

baseline = np.select(
    [segment == 'new', segment == 'regular', segment == 'vip'],
    [3.5, 7.5, 16.0]
)

true_effect = 0.8
noise = rng.normal(0.0, 5.5, n)
y = baseline + true_effect * variant + noise

exp = pd.DataFrame({
    'segment': segment,
    'variant': variant,
    'y': y,
})

exp.head()

,segment,variant,y
0,regular,0,0.5616
1,new,0,2.3353
2,new,0,6.4584
3,new,1,5.8533
4,new,1,4.0095


## Задача 1. Посчитать обычную оценку эффекта

Посчитайте naive estimator:

$$
\widehat{\tau}_{\mathrm{naive}}
=
\bar{Y}_{\mathrm{treatment}}
-
\bar{Y}_{\mathrm{control}}
$$

и стандартную ошибку:

$$
\operatorname{SE}(\widehat{\tau}_{\mathrm{naive}})
=
\sqrt{
\frac{s_{\mathrm{treatment}}^2}{n_{\mathrm{treatment}}}
+
\frac{s_{\mathrm{control}}^2}{n_{\mathrm{control}}}
}
$$

Нужно создать переменные:

- `effect_naive`
- `se_naive`

In [ ]:
treatment = exp['y'][variant == 1]
control = exp['y'][variant == 0]

effect_naive = treatment.mean() - control.mean()
variance = treatment.var(ddof=1) / len(treatment) + control.var(ddof=1) / len(control)
se_naive = np.sqrt(variance)

In [30]:
naive_tab = {
    'effect': effect_naive,
    'se': se_naive,
    'ci_left': effect_naive - 1.96 * se_naive,
    'ci_right': effect_naive + 1.96 * se_naive
}
naive_tab

{'effect': np.float64(0.7054958425752069),
 'se': np.float64(0.1039470212397462),
 'ci_left': np.float64(0.5017596809453044),
 'ci_right': np.float64(0.9092320042051094)}

In [17]:
assert 'effect_naive' in globals(), 'Создайте переменную effect_naive'
assert 'se_naive' in globals(), 'Создайте переменную se_naive'
assert np.isscalar(effect_naive), 'effect_naive должен быть числом'
assert np.isscalar(se_naive), 'se_naive должен быть числом'
assert se_naive > 0, 'se_naive должен быть положительным'

print('effect_naive =', round(effect_naive, 4))
print('se_naive =', round(se_naive, 4))

effect_naive = 0.7055
se_naive = 0.1039


## Задача 2. Посчитать таблицу по стратам

Для каждой страты `segment` посчитайте:

$$
\widehat{\tau}_{h}
=
\bar{Y}_{\mathrm{treatment}, h}
-
\bar{Y}_{\mathrm{control}, h}
$$

$$
w_h
=
\frac{N_h}{N}
$$

$$
\widehat{\operatorname{Var}}(\widehat{\tau}_{h})
=
\frac{s_{\mathrm{treatment},h}^2}{n_{\mathrm{treatment},h}}
+
\frac{s_{\mathrm{control},h}^2}{n_{\mathrm{control},h}}
$$

Нужно создать DataFrame `strata_table` со столбцами:

- `segment`
- `weight`
- `tau_h`
- `var_h`

**Требование:** сделайте расчет явно через цикл или через `groupby`, но без готовой функции для стратифицированной оценки.

In [ ]:
rows = []

for h, part in exp.groupby('segment'):
    treatment = part.loc[part['variant'] == 1, 'y']
    control = part.loc[part['variant'] == 0, 'y']

    weight_h = len(part) / len(exp)
    tau_h = treatment.mean() - control.mean()
    var_h = treatment.var(ddof=1) / len(treatment) + control.var(ddof=1) / len(control)

    rows.append(
        {
            'segment': h, 
            'weight': weight_h,
            'tau_h': tau_h,
            'var_h': var_h
        }
    )

strata_table = pd.DataFrame(rows)
strata_table

,segment,weight,tau_h,var_h
0,new,0.5538,0.8506,0.0123
1,regular,0.3002,0.6729,0.0219
2,vip,0.1460,0.3023,0.0461


In [ ]:
assert 'strata_table' in globals(), 'Создайте DataFrame strata_table'
assert isinstance(strata_table, pd.DataFrame), 'strata_table должен быть pandas DataFrame'
required_cols = {'segment', 'weight', 'tau_h', 'var_h'}
assert required_cols.issubset(strata_table.columns), f'В strata_table должны быть столбцы {required_cols}'
assert len(strata_table) == exp['segment'].nunique(), 'В таблице должна быть одна строка на каждую страту'
assert np.isclose(strata_table['weight'].sum(), 1.0), 'Сумма весов должна быть равна 1'
assert (strata_table['var_h'] > 0).all(), 'Все var_h должны быть положительными'

strata_table

## Задача 3. Собрать stratified estimator

Используйте формулу:

$$
\widehat{\tau}_{\mathrm{strat}}
=
\sum_{h=1}^{H} w_h\widehat{\tau}_{h}
$$

и стандартную ошибку:

$$
\operatorname{SE}(\widehat{\tau}_{\mathrm{strat}})
=
\sqrt{
\sum_{h=1}^{H} w_h^2
\widehat{\operatorname{Var}}(\widehat{\tau}_{h})
}
$$

Нужно создать переменные:

- `effect_strat`
- `se_strat`

In [25]:
effect_strat = np.sum(strata_table['weight']*strata_table['tau_h'])
var_strat = np.sum((strata_table['weight'] ** 2) * strata_table['var_h'])
se_strat = np.sqrt(var_strat)

In [33]:
strat_tab = {
    'effect': effect_strat,
    'se': se_strat,
    'ci_left': effect_strat - 1.96 * se_strat,
    'ci_right': effect_strat + 1.96 * se_strat
}
strat_tab

{'effect': np.float64(0.717245309534756),
 'se': np.float64(0.08204675742614698),
 'ci_left': np.float64(0.5564336649795079),
 'ci_right': np.float64(0.8780569540900041)}

In [34]:
assert 'effect_strat' in globals(), 'Создайте переменную effect_strat'
assert 'se_strat' in globals(), 'Создайте переменную se_strat'
assert np.isscalar(effect_strat), 'effect_strat должен быть числом'
assert np.isscalar(se_strat), 'se_strat должен быть числом'
assert se_strat > 0, 'se_strat должен быть положительным'

print('effect_strat =', round(effect_strat, 4))
print('se_strat =', round(se_strat, 4))

effect_strat = 0.7172
se_strat = 0.082


## Задача 4. Сравнить naive и stratified

Соберите таблицу `comparison` со строками для методов `naive` и `stratified`.

В таблице должны быть столбцы:

- `method`
- `effect`
- `se`
- `ci_left`
- `ci_right`
- `ci_width`

Доверительный интервал считайте как:

$$
\widehat{\tau} \pm 1.96 \cdot \operatorname{SE}(\widehat{\tau})
$$

In [36]:
comparison = pd.DataFrame([
    {'method': 'naive', **naive_tab},
    {'method': 'stratified', **strat_tab}
])
comparison['ci_width'] = comparison['ci_right'] - comparison['ci_left']

In [37]:
assert 'comparison' in globals(), 'Создайте DataFrame comparison'
assert isinstance(comparison, pd.DataFrame), 'comparison должен быть pandas DataFrame'
required_cols = {'method', 'effect', 'se', 'ci_left', 'ci_right', 'ci_width'}
assert required_cols.issubset(comparison.columns), f'В comparison должны быть столбцы {required_cols}'
assert set(comparison['method']) == {'naive', 'stratified'}, 'Должны быть методы naive и stratified'
assert (comparison['ci_width'] > 0).all(), 'Ширина доверительного интервала должна быть положительной'

comparison

,method,effect,se,ci_left,ci_right,ci_width
0,naive,0.7055,0.1039,0.5018,0.9092,0.4075
1,stratified,0.7172,0.0820,0.5564,0.8781,0.3216
